# ArchX - Fine-Tuning LoRA sur AMD Developer Cloud

Ce notebook fine-tune le modèle **Gemma 4 12B Unified** (`google/gemma-4-12B-it`)
avec **LoRA** (Low-Rank Adaptation) sur notre dataset d'analyses architecturales.

**Pipeline :**
1. Détection du GPU AMD et de la VRAM
2. Installation des dépendances
3. Chargement du dataset
4. Chargement du modèle (bf16 ou 4-bit selon VRAM)
5. Configuration LoRA
6. Entraînement avec SFTTrainer
7. Sauvegarde de l'adaptateur
8. Test d'inférence

## Cellule 1 — Installation des dépendances

In [ ]:
# Installation des bibliothèques nécessaires pour le fine-tuning sur AMD ROCm
# Note : On n'installe PAS bitsandbytes car il est instable sur ROCm.
#        On utilisera bf16 natif (parfaitement supporté sur MI200X/MI300X).
# Note Gemma 4 : nécessite transformers avec support day-one de Gemma 4 Unified
#        (google/gemma-4-12B-it, sorti le 03/06/2026 — classe AutoModelForMultimodalLM).
#        --upgrade force la dernière version publiée, indispensable ici.

import subprocess
import sys

packages = [
    'transformers',
    'trl',
    'peft',
    'accelerate',
    'datasets',
    'huggingface_hub',
]

for pkg in packages:
    print(f'Installation de {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg])

print('\n✅ Toutes les dépendances sont installées.')

## Cellule 2 — Détection du GPU AMD

In [ ]:
import subprocess
import torch

print('=' * 60)
print('        DÉTECTION DU GPU AMD')
print('=' * 60)

# --- Infos via rocm-smi (outil AMD natif) ---
print('\n📊 Informations rocm-smi :')
try:
    result = subprocess.run(['rocm-smi', '--showproductname', '--showmeminfo', 'vram'],
                           capture_output=True, text=True, timeout=10)
    print(result.stdout)
except FileNotFoundError:
    print('  ⚠️  rocm-smi non trouvé — peut-être amd-smi à la place.')
    try:
        result = subprocess.run(['amd-smi', 'metric', '--vram'],
                               capture_output=True, text=True, timeout=10)
        print(result.stdout)
    except Exception:
        print('  ⚠️  amd-smi aussi non trouvé. On continue avec torch.')

# --- Infos via PyTorch ---
print('\n🔥 PyTorch + ROCm :')
if torch.cuda.is_available():
    n_gpus = torch.cuda.device_count()
    print(f'  GPU(s) disponibles : {n_gpus}')
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / (1024**3)
        print(f'  GPU {i} : {props.name}')
        print(f'    VRAM totale : {vram_gb:.1f} GB')
        print(f'    Multiprocesseurs : {props.multi_processor_count}')

    # VRAM libre sur le GPU 0
    free_mem, total_mem = torch.cuda.mem_get_info(0)
    free_gb  = free_mem  / (1024**3)
    total_gb = total_mem / (1024**3)
    print(f'\n  VRAM GPU 0 — Libre : {free_gb:.1f} GB / Total : {total_gb:.1f} GB')

    # Décision automatique du mode de chargement du modèle
    if total_gb >= 40:
        LOAD_MODE = 'bf16'
        print(f'\n  ✅ VRAM suffisante ({total_gb:.0f} GB) → Chargement en bf16 (recommandé sur AMD).')
    else:
        LOAD_MODE = '4bit'
        print(f'\n  ⚠️  VRAM limitée ({total_gb:.0f} GB) → Chargement en 4-bit (bitsandbytes requis).')
else:
    print('  ❌ Aucun GPU détecté par PyTorch ! Vérifiez votre installation ROCm.')
    LOAD_MODE = 'cpu'

print(f'\n  MODE SÉLECTIONNÉ : {LOAD_MODE}')
print('=' * 60)

## Cellule 3 — Connexion Hugging Face

In [ ]:
# Gemma 4 12B (google/gemma-4-12B-it) est publié sous licence Apache 2.0 et n'est,
# a priori, PAS gated (contrairement à Gemma 2/3) — mais on garde ce login HF par
# sécurité/portabilité (ex. si vous repassez sur un modèle Gemma gated plus tard).
#   1. Avoir un compte Hugging Face
#   2. Créer un token d'accès sur : https://huggingface.co/settings/tokens

from huggingface_hub import login
import os

# Méthode 1 : Token dans une variable d'environnement (recommandé)
hf_token = os.environ.get('HF_TOKEN', '')

# Méthode 2 : Saisie manuelle si la variable n'est pas définie
if not hf_token:
    hf_token = input('🔑 Entrez votre token Hugging Face (hf_...) : ').strip()

login(token=hf_token, add_to_git_credential=False)
print('✅ Connexion Hugging Face réussie.')

## Cellule 4 — Chargement du dataset

In [ ]:
import json
from pathlib import Path
from datasets import Dataset
from transformers import AutoProcessor

# ─────────────────────────────────────────────────────────
# CONFIGURATION : Choisissez votre modèle de base
# (Défini ici, avant le dataset, car le processor/chat template de Gemma 4
#  est nécessaire pour construire le texte d'entraînement ci-dessous)
# ─────────────────────────────────────────────────────────
# Option A : Gemma 4 12B Unified (recommandé sur 192GB VRAM — remplace Gemma 3 12B-it)
BASE_MODEL = 'google/gemma-4-12B-it'

# Option B : Gemma 4 E4B (si VRAM limitée)
# BASE_MODEL = 'google/gemma-4-E4B-it'
# ─────────────────────────────────────────────────────────

# Chemin vers votre fichier training_data.jsonl — le notebook tourne dans
# training/, mais le dataset vit dans dataset_generation/ (dossier voisin
# dans le repo), d'où le '../'. Si votre training_data.jsonl est ailleurs
# (ex: copié à la racine sur cette VM), ajustez ce chemin en conséquence.
DATASET_PATH = '../dataset_generation/training_data.jsonl'

# Gemma 4 12B Unified est multimodal (texte+image+audio+vidéo, architecture
# "encoder-free") — on charge un AutoProcessor (pas un simple AutoTokenizer).
# Le tokenizer texte reste accessible via processor.tokenizer.
# Le chat template attend un contenu structuré en liste de "parts", ex :
#   {'role': 'user', 'content': [{'type': 'text', 'text': '...'}]}
# (identique à Gemma 3 pour la partie texte — on n'utilise ici que du texte,
# aucune image/audio, donc les pipelines vision/audio ne sont jamais sollicités)
#
# IMPORTANT — mode "thinking" : Gemma 4 supporte un mode de raisonnement activé
# en ajoutant un token de contrôle <|think|> en tête du system prompt. On ne
# l'ajoute PAS ici : on veut une sortie JSON directe, pas une trace de
# raisonnement. À VÉRIFIER en cellule 9 (test d'inférence) : certains variants
# émettent quand même un bloc de pensée VIDE par défaut
# (`<|channel>thought\n<channel|>`) — si c'est le cas ici, il faudra soit le
# strip avant json.loads, soit l'inclure tel quel dans les exemples
# d'entraînement pour rester cohérent entraînement/production (cf. le bug de
# format Gemma 3 déjà corrigé plus haut dans ce projet).
processor = AutoProcessor.from_pretrained(BASE_MODEL)
tokenizer = processor.tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'  # Nécessaire pour SFTTrainer

# IMPORTANT : on importe le MÊME prompt builder que la production
# (api/services/pipeline.py), au lieu d'en garder une version locale dupliquée.
# C'est le fix du bug "le modèle entraîné n'a jamais vu le format qu'il reçoit
# en prod" — le fichier training/format_for_training.py vit à côté de ce
# notebook (import direct, pas besoin du package `training.`).
from format_for_training import SYSTEM_PROMPT, build_user_message

def load_jsonl(path: str) -> list[dict]:
    """Charge un fichier JSONL ligne par ligne."""
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def format_as_chat(record: dict) -> str:
    """
    Convertit un exemple du dataset en texte de conversation via le chat template
    natif du processor — la cohérence avec le format vu à l'entraînement est cruciale.
    Utilise build_user_message/SYSTEM_PROMPT de format_for_training.py, exactement
    comme le fait api/services/pipeline.py en production.
    """
    lang = record.get('language', 'fr')
    user_msg = build_user_message(lang, record['input'])
    assistant_msg = json.dumps(record['output'], ensure_ascii=False)
    messages = [
        {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT[lang]}]},
        {'role': 'user', 'content': [{'type': 'text', 'text': user_msg}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_msg}]},
    ]
    return processor.apply_chat_template(messages, tokenize=False)

# Chargement
print(f'📂 Chargement du dataset depuis : {DATASET_PATH}')
raw_data = load_jsonl(DATASET_PATH)
print(f'  → {len(raw_data)} exemples chargés.')

# Formatage
formatted = [{'text': format_as_chat(r)} for r in raw_data]
dataset = Dataset.from_list(formatted)

# Séparation train / validation (90% / 10%)
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f'  → Entraînement : {len(train_dataset)} exemples')
print(f'  → Validation   : {len(eval_dataset)} exemples')
print('\n📄 Exemple de texte formaté (extrait) :')
print(train_dataset[0]['text'][:500] + '...')

## Cellule 5 — Chargement du modèle

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, BitsAndBytesConfig

# Gemma 4 12B Unified est multimodal (texte/image/audio/vidéo, architecture
# encoder-free) — la classe officiellement documentée pour le charger est
# AutoModelForMultimodalLM (pas AutoModelForCausalLM, ni un import direct de
# Gemma4UnifiedForConditionalGeneration — l'Auto class est plus stable dans le
# temps si Google renomme la classe interne).
print(f'🤖 Chargement du modèle : {BASE_MODEL}')
print(f'   Mode : {LOAD_MODE}')

# Chargement du modèle selon la VRAM disponible
if LOAD_MODE == 'bf16':
    # Mode recommandé sur AMD — bf16 est natif sur les cartes MI series
    model = AutoModelForMultimodalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map='auto',           # Répartit automatiquement sur le(s) GPU(s)
        attn_implementation='eager', # 'eager' est plus stable que 'flash_attention_2' sur ROCm
    )
    print('✅ Modèle chargé en bf16.')

elif LOAD_MODE == '4bit':
    # Mode 4-bit pour GPU avec moins de VRAM (nécessite bitsandbytes)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    model = AutoModelForMultimodalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map='auto',
    )
    print('✅ Modèle chargé en 4-bit (QLoRA).')

else:
    raise EnvironmentError('Aucun GPU disponible. Fine-tuning impossible sans GPU.')

# Affichage de la VRAM utilisée après chargement
torch.cuda.synchronize()
used_gb = torch.cuda.memory_allocated(0) / (1024**3)
print(f'   VRAM utilisée après chargement du modèle : {used_gb:.1f} GB')

## Cellule 6 — Configuration LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Préparation du modèle pour le fine-tuning en 4-bit (si applicable)
if LOAD_MODE == '4bit':
    model = prepare_model_for_kbit_training(model)

# Activation du gradient checkpointing pour réduire la consommation mémoire
# (Permet d'entraîner des modèles plus larges en échange d'un peu de vitesse)
model.gradient_checkpointing_enable()

# ─────────────────────────────────────────────────────────
# CONFIGURATION LORA
# r=16, alpha=32 : bon équilibre entre puissance d'adaptation et généralisation
# (Avec 192GB de VRAM vous avez la marge pour monter à r=32 si vous voulez plus
# de capacité d'adaptation — on garde r=16 par défaut pour rester comparable
# au run Gemma 3 déjà validé sur ce dataset.)
#
# Note Gemma 4 : ces noms de projection (q_proj, gate_proj, ...) sont ceux du
# décodeur texte (Gemma4UnifiedTextModel). Le modèle complet
# (AutoModelForMultimodalLM / Gemma4UnifiedForConditionalGeneration) est
# multimodal, donc ces couches sont nichées sous un préfixe (ex:
# model.language_model.layers.*), comme c'était déjà le cas avec Gemma 3.
# PEFT matche par nom de module en suffixe donc ça devrait fonctionner tel
# quel ; si "trainable%" ci-dessous affiche 0%, inspectez
# `model.named_modules()`.
# ─────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                        # Rang de la décomposition — plus élevé = plus de paramètres appris
    lora_alpha=32,               # Facteur de scaling (généralement 2x le rang)
    target_modules=[             # Modules de l'attention sur lesquels appliquer LoRA
        'q_proj',                # Query projection
        'k_proj',                # Key projection
        'v_proj',                # Value projection
        'o_proj',                # Output projection
        'gate_proj',             # MLP gate
        'up_proj',               # MLP up
        'down_proj',             # MLP down
    ],
    lora_dropout=0.05,           # Régularisation pour éviter le surapprentissage
    bias='none',                 # Pas de biais entraînable (standard)
    task_type=TaskType.CAUSAL_LM # Modèle de langage causal (génération de texte)
)

# Application de la configuration LoRA au modèle
model = get_peft_model(model, lora_config)

# Affichage du nombre de paramètres entraînables vs total
model.print_trainable_parameters()
# Résultat attendu : seulement ~1-2% des paramètres sont entraînables avec LoRA
print('\n✅ LoRA configuré et appliqué.')

## Cellule 7 — Entraînement avec SFTTrainer

In [ ]:
import math
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

# Dossier de sauvegarde du modèle fine-tuné
OUTPUT_DIR = './architect-insight-lora'

# warmup_ratio est déprécié (retrait prévu en transformers v5.2) au profit de
# warmup_steps — on calcule l'équivalent en steps à partir de la taille du dataset
# (3% du nombre total de steps d'entraînement, comme avant).
_NUM_EPOCHS = 3
_BATCH_SIZE = 8     # 192GB VRAM permet un batch par device bien plus gros qu'en 48GB (était 2)
_GRAD_ACCUM = 1     # Effective batch = 8, identique à avant (2*4) mais moins d'overhead d'accumulation
_steps_per_epoch = math.ceil(len(train_dataset) / (_BATCH_SIZE * _GRAD_ACCUM))
_total_steps = _steps_per_epoch * _NUM_EPOCHS
_warmup_steps = max(1, int(0.03 * _total_steps))

# ─────────────────────────────────────────────────────────
# HYPERPARAMÈTRES D'ENTRAÎNEMENT
# Ajustés pour AMD ROCm et pour notre taille de dataset (~160-250 exemples)
# ─────────────────────────────────────────────────────────
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Epochs et taille de batch ──
    num_train_epochs=_NUM_EPOCHS,     # 3 epochs = bon compromis pour ~200 exemples
    per_device_train_batch_size=_BATCH_SIZE,   # 192GB VRAM → augmenté par rapport au run Gemma 3 (48GB)
    per_device_eval_batch_size=_BATCH_SIZE,
    gradient_accumulation_steps=_GRAD_ACCUM,   # Batch effectif = 8 * 1 = 8 exemples

    # ── Optimiseur ──
    optim='adamw_torch',             # 'adamw_torch' est stable sur ROCm
    learning_rate=2e-4,              # Taux d'apprentissage standard pour LoRA
    lr_scheduler_type='cosine',      # Scheduler cosinus (décroissance douce)
    warmup_steps=_warmup_steps,      # Remplace warmup_ratio=0.03 (déprécié)
    weight_decay=0.01,               # Légère régularisation L2

    # ── Précision et mémoire ──
    bf16=True,                       # bf16 est le mode natif des GPU AMD MI series
    fp16=False,                      # Ne pas activer les deux en même temps
    gradient_checkpointing=True,     # Économie mémoire
    dataloader_num_workers=0,        # 0 = utilise le thread principal (plus stable sur ROCm)

    # ── Longueur de séquence ──
    max_length=2048,                 # Longueur max des tokens (nos exemples font ~600-1200 tokens)
                                      # Renommé depuis max_seq_length (trl >= 0.20)
    dataset_text_field='text',       # Nom du champ texte dans notre dataset
    packing=False,                   # False = plus simple, True = plus rapide mais plus complexe

    # ── Évaluation et sauvegarde ──
    eval_strategy='steps',
    eval_steps=50,                   # Évaluer toutes les 50 étapes
    save_strategy='steps',
    save_steps=50,
    save_total_limit=2,              # Garder seulement les 2 meilleurs checkpoints
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',

    # ── Logs ──
    logging_steps=10,
    # logging_dir est déprécié (retrait v5.2) — sans effet ici de toute façon car
    # report_to='none' (pas de TensorBoard/W&B). Pour réactiver TensorBoard plus tard,
    # définir la variable d'env TENSORBOARD_LOGGING_DIR et passer report_to='tensorboard'.
    report_to='none',                # 'none' = pas de Weights & Biases ni TensorBoard externe

    # ── Reproductibilité ──
    seed=42,
)

# Création du trainer
# Note : trl a renommé l'argument `tokenizer=` en `processing_class=` dans les
# versions récentes (installées via --upgrade en cellule 1).
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

print('🏋️ Démarrage de l\'entraînement...')
print(f'   Exemples d\'entraînement : {len(train_dataset)}')
print(f'   Epochs : {training_args.num_train_epochs}')
print(f'   Batch effectif : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}')
print('─' * 50)

# Lancement de l'entraînement
train_result = trainer.train()

print('\n' + '─' * 50)
print('✅ Entraînement terminé !')
print(f'   Loss finale d\'entraînement : {train_result.training_loss:.4f}')
print('   (Un loss < 0.8 est un bon signe. Un loss > 2.0 indique un problème.)')

## Cellule 8 — Sauvegarde de l'adaptateur LoRA

In [ ]:
import os

# Sauvegarde de l'adaptateur LoRA (SEULEMENT les poids supplémentaires, ~100 Mo)
# Ce n'est PAS le modèle complet — c'est uniquement le "delta" appris par LoRA.
# Pour utiliser le modèle, il faudra charger Gemma de base + cet adaptateur.

ADAPTER_SAVE_PATH = './architect-insight-lora-final'

print(f'💾 Sauvegarde de l\'adaptateur LoRA vers : {ADAPTER_SAVE_PATH}')
trainer.model.save_pretrained(ADAPTER_SAVE_PATH)
processor.save_pretrained(ADAPTER_SAVE_PATH)  # Sauvegarde aussi le tokenizer (processor.tokenizer)

# Vérification
saved_files = os.listdir(ADAPTER_SAVE_PATH)
print(f'   Fichiers sauvegardés : {saved_files}')

# Taille de l'adaptateur
total_size = sum(
    os.path.getsize(os.path.join(ADAPTER_SAVE_PATH, f))
    for f in saved_files
) / (1024**2)
print(f'   Taille totale de l\'adaptateur : {total_size:.1f} MB')

print('\n✅ Adaptateur LoRA sauvegardé avec succès !')
print('   Pour réutiliser le modèle fine-tuné plus tard :')
print(f'   model = PeftModel.from_pretrained(base_model, \'{ADAPTER_SAVE_PATH}\')')

## Cellule 8bis — Fusion et sauvegarde du modèle COMPLET (pour la production)

⚠️ **Indispensable pour le déploiement.** `api/services/pipeline.py::call_model`
charge `ARCHX_MODEL_PATH` via `AutoModelForMultimodalLM.from_pretrained(model_path, ...)`
— ça suppose un modèle complet, pas juste l'adaptateur LoRA (~100 Mo) sauvegardé en
cellule 8. Sans cette étape, pointer `ARCHX_MODEL_PATH` vers `architect-insight-lora-final`
échouera silencieusement et l'API retombera en mode mock pendant la démo.

In [5]:
import os

# Cellule autonome : ne dépend pas de `trainer`/`model` en mémoire (utile si le
# kernel a redémarré après l'entraînement) — recharge tout depuis le disque à
# partir de l'adaptateur sauvegardé en cellule 8.

ADAPTER_SAVE_PATH = './architect-insight-lora-final'
MERGED_MODEL_PATH = './architect-insight-gemma4-merged'
BASE_MODEL = 'google/gemma-4-12B-it'  # doit correspondre au modèle utilisé à l'entraînement

if not os.path.isdir(ADAPTER_SAVE_PATH) or not os.listdir(ADAPTER_SAVE_PATH):
    raise FileNotFoundError(
        f"'{ADAPTER_SAVE_PATH}' est introuvable ou vide — l'adaptateur LoRA n'a "
        f"jamais été sauvegardé sur le disque (cellule 8 pas exécutée avant le "
        f"redémarrage du kernel, ou entraînement pas terminé). Il faut relancer "
        f"les cellules 4 → 8 pour ré-entraîner avant de pouvoir fusionner/sauvegarder."
    )

print(f'📂 Adaptateur trouvé : {ADAPTER_SAVE_PATH} ({os.listdir(ADAPTER_SAVE_PATH)})')

import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor
from peft import PeftModel

print(f'🤖 Rechargement du modèle de base : {BASE_MODEL}...')
base_model = AutoModelForMultimodalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    attn_implementation='eager',
)
processor = AutoProcessor.from_pretrained(ADAPTER_SAVE_PATH)  # tokenizer sauvegardé avec l'adaptateur

print('🔌 Application de l\'adaptateur LoRA...')
model_with_adapter = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)

print('🔀 Fusion de l\'adaptateur dans le modèle de base (merge_and_unload)...')
print('   (quelques minutes, ~24 Go en bf16 pour un modèle 12B)')
merged_model = model_with_adapter.merge_and_unload()
merged_model.save_pretrained(MERGED_MODEL_PATH, safe_serialization=True)
processor.save_pretrained(MERGED_MODEL_PATH)

saved_files = os.listdir(MERGED_MODEL_PATH)
total_size_gb = sum(
    os.path.getsize(os.path.join(MERGED_MODEL_PATH, f))
    for f in saved_files
) / (1024**3)

print(f'\n✅ Modèle complet sauvegardé dans : {MERGED_MODEL_PATH}')
print(f'   Fichiers : {saved_files}')
print(f'   Taille totale : {total_size_gb:.1f} GB')
print(f'\n   Pour la production, pointez ARCHX_MODEL_PATH vers CE dossier')
print(f'   (pas vers architect-insight-lora-final, qui ne contient que l\'adaptateur) :')
print(f'   ARCHX_MODEL_PATH={os.path.abspath(MERGED_MODEL_PATH)}')

# ─────────────────────────────────────────────────────────
# OPTIONNEL mais recommandé : sauvegarde durable sur Hugging Face Hub.
# Jupiter est une instance louée/éphémère — si vous voulez garder ce modèle
# au-delà de la durée de location (ou le charger depuis une autre machine de
# serving plus tard, cf. Piste B du plan de déploiement), poussez-le sur le Hub :
#
REPO_ID = 'Karmelkke/archx-gemma4-12b-merged'  # à adapter
merged_model.push_to_hub(REPO_ID, private=True)
processor.push_to_hub(REPO_ID, private=True)
print(f'✅ Poussé sur https://huggingface.co/{REPO_ID}')
print(f'   ARCHX_MODEL_PATH peut alors être directement "{REPO_ID}" (repo id),')
print(f'   from_pretrained() saura le télécharger depuis le Hub.')
# ─────────────────────────────────────────────────────────

📂 Adaptateur trouvé : ./architect-insight-lora-final (['README.md', 'tokenizer_config.json', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors', 'processor_config.json', 'chat_template.jinja'])
🤖 Rechargement du modèle de base : google/gemma-4-12B-it...


Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

🔌 Application de l'adaptateur LoRA...
🔀 Fusion de l'adaptateur dans le modèle de base (merge_and_unload)...
   (quelques minutes, ~24 Go en bf16 pour un modèle 12B)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✅ Modèle complet sauvegardé dans : ./architect-insight-gemma4-merged
   Fichiers : ['config.json', 'tokenizer_config.json', 'tokenizer.json', 'generation_config.json', 'model.safetensors', 'processor_config.json', 'chat_template.jinja']
   Taille totale : 22.3 GB

   Pour la production, pointez ARCHX_MODEL_PATH vers CE dossier
   (pas vers architect-insight-lora-final, qui ne contient que l'adaptateur) :
   ARCHX_MODEL_PATH=/shared-docker/ArchX/training/architect-insight-gemma4-merged


...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Poussé sur https://huggingface.co/Karmelkke/archx-gemma4-12b-merged
   ARCHX_MODEL_PATH peut alors être directement "Karmelkke/archx-gemma4-12b-merged" (repo id),
   from_pretrained() saura le télécharger depuis le Hub.


## Cellule 9 — Test d'inférence

In [ ]:
import json
import re
import torch
from transformers import pipeline

print('🧪 Test d\'inférence avec le modèle fine-tuné...')
print('─' * 60)

# Mettre le modèle en mode évaluation
model.eval()

# ─────────────────────────────────────────────────────────
# SCÉNARIO DE TEST : Un projet e-commerce avec des problèmes
# Volontairement SANS language/database_name — c'est exactement le scénario
# qui a révélé le bug de hallucination de stack (le modèle inventait
# ".NET/ASP.NET Core with SQL Server"). Sans ces clés, le modèle corrigé ne
# doit plus mentionner aucune stack technique dans project_description.
# ─────────────────────────────────────────────────────────
test_metrics = {
    'sector': 'e-commerce',
    'team_size': 8,
    'metrics': {
        'coupling_score': 7.2,           # Couplage élevé — mauvais signe
        'cohesion_score': 3.4,           # Faible cohésion — spaghetti code
        'avg_cyclomatic_complexity': 12, # Complexité très élevée
        'test_coverage_estimate': 18.0,  # Tests insuffisants
        'top_hotspot_bugfix_ratio': 0.45 # 45% des commits corrigent des bugs
    },
    'anti_patterns': [
        {'type': 'God Class', 'location': 'src/ecommerce/handler.py', 'severity': 'high'},
        {'type': 'Long Method', 'location': 'src/ecommerce/service.py', 'severity': 'medium'},
        {'type': 'Circular Dependency', 'location': 'src/ecommerce/core.py', 'severity': 'medium'},
    ],
}

# Construction du prompt via le même builder que l'entraînement (cellule 4) ET
# la production (api/services/pipeline.py) — build_user_message + SYSTEM_PROMPT
# importés de format_for_training.py.
LANG = 'fr'
user_msg = build_user_message(LANG, test_metrics)
messages = [
    {'role': 'system', 'content': [{'type': 'text', 'text': SYSTEM_PROMPT[LANG]}]},
    {'role': 'user', 'content': [{'type': 'text', 'text': user_msg}]},
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors='pt',
).to(model.device, dtype=torch.bfloat16)

print('📝 Métriques envoyées au modèle :')
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))
print('\n🤖 Réponse du modèle fine-tuné :')
print('─' * 60)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,    # Longueur max de la réponse générée
        temperature=0.1,        # Très bas = réponse déterministe et précise (bon pour JSON)
        do_sample=True,
        repetition_penalty=1.1, # Évite les répétitions
        pad_token_id=tokenizer.eos_token_id,
    )

# Décodage de la réponse (en ignorant le prompt d'entrée)
response = tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)

print(response)

# Gemma 4 : on n'a pas demandé le mode "thinking" (pas de <|think|> dans le
# system prompt), mais certains variants émettent quand même un bloc de
# pensée vide par défaut (`<|channel>thought ... <channel|>`). On le retire
# avant de parser le JSON — s'il apparaît ICI de façon non vide, c'est le
# signe qu'il faudra revoir le format des exemples d'entraînement pour
# rester cohérent avec ce que le modèle produit réellement en génération.
thinking_match = re.search(r'<\|channel\|?>thought.*?<channel\|>', response, re.DOTALL)
if thinking_match:
    print(f'\n⚠️  Bloc de pensée détecté dans la sortie brute : {thinking_match.group()[:200]!r}')
response_for_json = re.sub(r'<\|channel\|?>thought.*?<channel\|>', '', response, flags=re.DOTALL).strip()

# Vérification que la réponse est un JSON valide
print('\n─' * 60)
try:
    parsed = json.loads(response_for_json)
    print('✅ La réponse est un JSON valide !')
    print(f'   Recommandation : {parsed.get("recommendation", "?")}')
    # Garde-fou anti-hallucination : la description ne doit citer aucune stack
    # technique puisqu'aucune n'a été fournie dans test_metrics ci-dessus.
    desc = parsed.get('project_description', '')
    suspects = ['.NET', 'ASP.NET', 'SQL Server', 'Django', 'Spring', 'Rails', 'Laravel']
    hallucinated = [s for s in suspects if s.lower() in desc.lower()]
    if hallucinated:
        print(f'⚠️  ATTENTION : project_description mentionne une stack non fournie : {hallucinated}')
    else:
        print('✅ Aucune stack technique inventée dans project_description.')
except json.JSONDecodeError as e:
    print(f'⚠️  La réponse n\'est pas un JSON parfait : {e}')
    print('   (Normal lors des premières inférences — le modèle peut parfois ajouter du texte avant le JSON)')

## Cellule 10 — (Optionnel) Benchmark avant/après

Cette cellule compare les réponses du modèle **avant** et **après** le fine-tuning sur un même jeu de test.
C'est une démonstration puissante pour le jury du concours AMD !


In [ ]:
# Pour charger le modèle de base SANS fine-tuning et comparer les réponses :

# from transformers import AutoModelForMultimodalLM
# from peft import PeftModel
# import torch

# base_model = AutoModelForMultimodalLM.from_pretrained(
#     BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto'
# )
# finetuned_model = PeftModel.from_pretrained(base_model, './architect-insight-lora-final')

# Générez avec les deux modèles sur le même prompt et comparez.
# Un bon fine-tuning devrait produire :
#   - Base model    → Texte libre, souvent non-JSON, verbeux
#   - Fine-tuné     → JSON structuré, précis, respectant vos garde-fous

print('Décommentez le code ci-dessus pour faire le benchmark avant/après.')
print('C\'est une slide de démonstration très percutante pour le jury AMD !')